# Build a small RNN to generate smiles codes

Try on two different datasets: a small toy dataset and the smiles column from the ESOL dataset (random sample of 300 points - training time estimated to be ~2-5 min - decrease the sample number if you had issues with runtime before).

In [79]:
import pandas as pd

smiles_list = [
    "CCO",
    "CCN",
    "C=O",
    "C1CCCCC1",
    "CC(=O)O",
    "CCC",
    "COC",
    "CCCl",
    "O"
]

df = pd.read_csv("esol_modified.csv").dropna(subset=["SMILES"]).sample(300)
smiles_list2 = df["SMILES"].to_list()

Build vocabulary and tokeniser

In [ ]:
chars = sorted(list(set("".join(smiles_list2))))
stoi = {ch: i for i, ch in enumerate(chars)} #string to index
itos = {i: ch for ch, i in stoi.items()} # index to string - to make output human readable

vocab_size = len(chars)

In [ ]:
print("stoi: ", stoi) # values are increasing. model will give more weight to characters with higher value. we need OHE
print("itos:", itos)

stoi:  {' ': 0, '#': 1, '(': 2, ')': 3, '/': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '=': 13, 'B': 14, 'C': 15, 'F': 16, 'H': 17, 'I': 18, 'N': 19, 'O': 20, 'P': 21, 'S': 22, '[': 23, '\\': 24, ']': 25, 'c': 26, 'l': 27, 'n': 28, 'o': 29, 'r': 30, 's': 31}
itos: {0: ' ', 1: '#', 2: '(', 3: ')', 4: '/', 5: '1', 6: '2', 7: '3', 8: '4', 9: '5', 10: '6', 11: '7', 12: '8', 13: '=', 14: 'B', 15: 'C', 16: 'F', 17: 'H', 18: 'I', 19: 'N', 20: 'O', 21: 'P', 22: 'S', 23: '[', 24: '\\', 25: ']', 26: 'c', 27: 'l', 28: 'n', 29: 'o', 30: 'r', 31: 's'}


In [82]:
def encode(smile):
    return [stoi[c] for c in smile]

data = [encode(s) for s in smiles_list2]

In [83]:
data

[[15,
  27,
  26,
  5,
  26,
  26,
  2,
  26,
  2,
  15,
  27,
  3,
  26,
  2,
  15,
  27,
  3,
  26,
  5,
  15,
  27,
  3,
  26,
  6,
  26,
  26,
  2,
  15,
  27,
  3,
  26,
  2,
  15,
  27,
  3,
  26,
  2,
  15,
  27,
  3,
  26,
  6,
  15,
  27],
 [15, 26, 5, 28, 26, 28, 26, 6, 28, 26, 26, 28, 26, 5, 6],
 [15,
  27,
  15,
  2,
  13,
  15,
  2,
  15,
  27,
  3,
  15,
  2,
  13,
  15,
  2,
  15,
  27,
  3,
  15,
  27,
  3,
  15,
  27,
  3,
  15,
  27],
 [15, 15, 15, 15, 15, 15, 15, 15, 15, 2, 13, 20, 3, 20, 15, 15],
 [15, 26, 5, 26, 26, 2, 15, 3, 26, 2, 20, 3, 26, 2, 15, 3, 26, 5],
 [15,
  27,
  26,
  5,
  26,
  26,
  26,
  2,
  26,
  26,
  5,
  3,
  26,
  6,
  26,
  26,
  26,
  26,
  2,
  15,
  27,
  3,
  26,
  6,
  15,
  27,
  0],
 [15, 5, 15, 15, 15, 15, 15, 15, 15, 5],
 [19,
  15,
  2,
  13,
  19,
  3,
  19,
  22,
  2,
  13,
  20,
  3,
  2,
  13,
  20,
  3,
  26,
  5,
  26,
  26,
  26,
  2,
  19,
  3,
  26,
  26,
  5,
  0],
 [26, 5, 26, 26, 28, 28, 26, 5],
 [15, 15, 15, 2, 13, 15, 

In [84]:
data = [seq for seq in data if len(seq) > 1]

Build Model

In [85]:
import torch
import torch.nn as nn

class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x) # two values are returned by RNN: output and hidden state (not used)
        out = self.fc(out)
        return out

Train the RNN

In [86]:
model = SmilesRNN(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    total_loss = 0

    for seq in data:
        x = torch.tensor(seq[:-1], dtype=torch.long).unsqueeze(0)
        y = torch.tensor(seq[1:]).unsqueeze(0)

        output = model(x)
        loss = criterion(output.view(-1, vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.3f}")

Epoch 0, Loss: 490.482
Epoch 20, Loss: 393.600
Epoch 40, Loss: 392.759
Epoch 60, Loss: 400.906
Epoch 80, Loss: 395.033
Epoch 100, Loss: 401.799
Epoch 120, Loss: 399.826
Epoch 140, Loss: 403.782
Epoch 160, Loss: 406.985
Epoch 180, Loss: 405.489


In [90]:
import random

def generate(model, start_char='C', max_len=8):
    model.eval()
    
    idx = torch.tensor([[stoi[start_char]]])
    result = start_char

    for i in range(max_len):
        output = model(idx)
        probs = torch.softmax(output[0, -1], dim=0)
        
        idx_next = torch.multinomial(probs, 1).item()
        char = itos[idx_next]

        result += char
        idx = torch.tensor([[idx_next]])

    return result

for i in range(10):
    print(generate(model)+"\n")

CCCCCCCOc

CCCCNc1)c

CCCCCCl)c

CCl)c1=Oc

CNc1=OCCC

CCCCC=OCN

Cl)C23(Nc

CC=O=Oc1C

CCc1=O=O=

CCCNc1=Oc



Some outputs with subset 'smiles_list' (not ESOL dataset):

C1CCOC=OC

COCCOCCOC

CC1COCOC=

CC=OCCOC=

CCCCCCOC=

CCCCCCCC=

CC=OCCCCC

CCO=O)OCC

C=OCCOCCO

CO)OCCC1C

CC=OC1COC

C1CCC=OCC

CCCCOCCCC

COCC=OCC=

C1CCC=OCC

CCCC=OCCC

C=O)OCCOC

C=OCCCCOC

CCCCCOCOC

C=OCCC=OC

CCCCC=OC1

COCC=OCCC

CCC1C=OC=

CCC1CCOCO

CCCCOCC=O

C=OCCCOC1

C=OCC=OC=

CCCCCCC1C

CCOCCOCCO

COCOC=OC=


with ESOL dataset sample of 300 entries:

CCCCCCc1=

COc3c1=Oc

Cl)c1=Oc1

CCCCCOCCC

Cl)Nc1=Oc

CCCCCCNSc

C123c1=O=

COc34CCCC

Cl)c1=OCC

CCCCO)c1N

CCCCCCCOc

CCCCNc1)c

CCCCCCl)c

CCl)c1=Oc

CNc1=OCCC

CCCCC=OCN

Cl)C23(Nc

CC=O=Oc1C

CCc1=O=O=

CCCNc1=Oc

so its not better at all ...